In [ ]:
# 1. Montar Drive (seguro e idempotente)
from google.colab import drive
import os, subprocess

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

Mounted at /content/drive


In [ ]:
# 2. Resolver caminhos
ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
RAW_PATH = os.path.join(BASE_PATH, "data_raw")
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
os.makedirs(PROC_PATH, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH :", RAW_PATH)
print("PROC_PATH:", PROC_PATH)

BASE_PATH: /content/drive/MyDrive/TCC_USP
RAW_PATH : /content/drive/MyDrive/TCC_USP/data_raw
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [ ]:
# 3. Carregar IBOV
import pandas as pd, os
ibov_file = os.path.join(RAW_PATH, "ibovespa.csv")
assert os.path.exists(ibov_file), f"Arquivo não encontrado: {ibov_file}"
df_ibov = pd.read_csv(ibov_file)
df_ibov["data"] = pd.to_datetime(df_ibov["data"])
print("IBOV:", df_ibov.shape)
display(df_ibov.head())

IBOV: (20, 2)


,data,close
0,2024-01-02,130749.73
1,2024-01-03,130673.55
2,2024-01-04,131624.44
3,2024-01-05,133734.42
4,2024-01-08,133528.27


In [ ]:
# 4. Carregar Notícias
news_file = os.path.join(RAW_PATH, "noticias_exemplo.csv")
assert os.path.exists(news_file), f"Arquivo não encontrado: {news_file}"
df_news = pd.read_csv(news_file)
df_news["data"] = pd.to_datetime(df_news["data"])
print("NEWS:", df_news.shape)
display(df_news.head())

NEWS: (10, 5)


,data,fonte,titulo,texto,sentimento
0,2024-01-02,exemplo,Titulo 1,Ibovespa sobe com otimismo no mercado internac...,1
1,2024-01-03,exemplo,Titulo 2,Bolsa cai após dados fracos da economia chinesa.,0
2,2024-01-04,exemplo,Titulo 3,Petróleo em alta puxa ações da Petrobras para ...,1
3,2024-01-05,exemplo,Titulo 4,Queda do dólar beneficia empresas exportadoras.,1
4,2024-01-08,exemplo,Titulo 5,Setor bancário recua com aversão ao risco.,0


In [ ]:
# 5. Pré-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))

def preprocess_text(t):
    t = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Usar a coluna 'texto'
df_news["clean_text"] = df_news["texto"].apply(preprocess_text)

print("✅ Pré-processamento concluído!")
display(df_news[["texto", "clean_text"]].head())

✅ Pré-processamento concluído!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,texto,clean_text
0,Ibovespa sobe com otimismo no mercado internac...,ibovespa sobe otimismo mercado internacional
1,Bolsa cai após dados fracos da economia chinesa.,bolsa cai após dados fracos economia chinesa
2,Petróleo em alta puxa ações da Petrobras para ...,petróleo alta puxa ações petrobras cima
3,Queda do dólar beneficia empresas exportadoras.,queda dólar beneficia empresas exportadoras
4,Setor bancário recua com aversão ao risco.,setor bancário recua aversão risco


In [38]:
# 6. Salvar processados
ibov_out  = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_out  = os.path.join(PROC_PATH, "noticias_clean.csv")
df_ibov.to_csv(ibov_out, index=False)
df_news.to_csv(news_out, index=False)

print("Salvo em data_processed:")
!ls -lh "{PROC_PATH}"

Salvo em data_processed:
total 2.0K
-rw------- 1 root root  427 Sep 20 21:47 ibovespa_clean.csv
-rw------- 1 root root 1.3K Sep 20 21:47 noticias_clean.csv
